In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import os

from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam


e:\CS 180 CNN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# load data
path = r"C:\Users\Robin\.cache\kagglehub\datasets\antorbosuantu\asa-real-fake-dataset\versions\4\Final Dataset"
train_path = path + r"\Train"
test_path  = path + r"\Test"

img_size = (224, 224)
batch_size = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=img_size,
    batch_size=batch_size
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 8000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
Classes: ['Train_Fake', 'Train_Real']


In [14]:
# augment data
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
).cache().prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y)
).prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [15]:
#setup ResNET-50
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

In [17]:
# run model
optimizer = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()
epochs = 10

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=epochs
)

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_6 (Sequential)   (None, 224, 224, 3)       0         
                                                                 
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d_4   (None, 2048)             0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dense_12 (Dense)            (None, 256)               524544    
                                                                 
 dropout_8 (Dropout)         (None, 256)               0         
                                                                 
 dense_13 (Dense)            (None, 128)               32896     
                                                      

In [18]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-6)
]

history_finetune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
125/125 [==============================] - 150s 1s/step - loss: 0.4484 - accuracy: 0.8024 - val_loss: 0.4953 - val_accuracy: 0.7795 - lr: 1.0000e-05
Epoch 2/10
125/125 [==============================] - 150s 1s/step - loss: 0.3297 - accuracy: 0.8505 - val_loss: 0.4354 - val_accuracy: 0.8140 - lr: 1.0000e-05
Epoch 3/10
125/125 [==============================] - 144s 1s/step - loss: 0.2894 - accuracy: 0.8679 - val_loss: 0.4763 - val_accuracy: 0.8015 - lr: 1.0000e-05
Epoch 4/10
125/125 [==============================] - 134s 1s/step - loss: 0.2628 - accuracy: 0.8849 - val_loss: 0.4536 - val_accuracy: 0.8190 - lr: 1.0000e-05
Epoch 5/10
125/125 [==============================] - 134s 1s/step - loss: 0.2437 - accuracy: 0.8959 - val_loss: 0.4605 - val_accuracy: 0.8135 - lr: 3.0000e-06


In [19]:
y_true = np.concatenate([y for x, y in test_ds])
y_pred = model.predict(test_ds)
y_pred = (y_pred > 0.5).astype(int)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

model.save("Veritas_Model_Apr29.h5")

32/32 [==============================] - 43s 1s/step
[[773 227]
 [145 855]]
              precision    recall  f1-score   support

           0       0.84      0.77      0.81      1000
           1       0.79      0.85      0.82      1000

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

